In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output
from dash.exceptions import PreventUpdate
import plotly.graph_objects as go

# Data / utilities
import base64
import pandas as pd

# JupyterDash helper
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import matplotlib.pyplot as plt


# CRUD module
from animal_shelter import AnimalShelter

###########################
# Data Manipulation / Model
###########################

username = "drewlogan"
password = "SNHU1234"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
docs_all = db.read({})
df = pd.DataFrame.from_records(docs_all)

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
if "_id" in df.columns:
    df.drop(columns=['_id'],inplace=True)

# Set of columns for the table
TABLE_COLUMNS = list(df.columns)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Load and embed the logo into the HTML to avoid file serving issues.
image_filename = 'Grazioso Salvare Logo.png'
with open(image_filename, "rb") as f:
    encoded_image = base64.b64encode(f.read()).decode("utf-8")
logo_src = f"data:image/png;base64,{encoded_image}"


def empty_figure(title: str) -> go.Figure:
    """Return a simple empty figure with a title"""
    fig = go.Figure()
    fig.update_layout(title=title)
    return fig


app.layout = html.Div(
    [
        # Header / Branding
        html.Center(
            html.Div(
                [
                    # Logo links to SNHU
                    html.A(
                        href="https://www.snhu.edu",
                        target="_blank",
                        children=html.Img(
                            src=logo_src,
                            style={"height": "90px", "marginBottom": "10px"},
                        ),
                    ),
                    html.H1("CS-340 Dashboard", style={"marginBottom": "10px"}),
                    html.A(
                        "SNHU Home Page",
                        href="https://www.snhu.edu",
                        target="_blank",
                        style={"display": "block", "marginTop": "4px"},
                    ),
                    html.Div(
                        "By: Drew Logan",
                        style={"fontWeight": "bold", "marginTop": "6px"},
                    ),
                ]
            )
        ),
    
        html.Hr(),
    
        # Interactive Filtering Options
        html.Div(
            [
                html.H4("Interactive Filter Options"),
                dcc.RadioItems(
                    id="rescue-filter",
                    options=[
                        {"label": "Water Rescue", "value": "WATER"},
                        {"label": "Mountain / Wilderness Rescue", "value": "MOUNTAIN"},
                        {"label": "Disaster Rescue / Individual Tracking", "value": "DISASTER"},
                        {"label": "Reset", "value": "RESET"},
                    ],
                    value="RESET",
                    labelStyle={"display": "inline-block", "marginRight": "16px"},
                    style={"marginBottom": "10px"},
                ),
                html.Div(
                    id="filter-status",
                    style={"fontStyle": "italic", "marginTop": "6px"},
                ),
            ]
        ),

        html.Hr(),
    
        # Data Table
        dash_table.DataTable(
            id='datatable-id',
            columns=[
                {"name": col, "id": col, "deletable": False, "selectable": True} 
                for col in TABLE_COLUMNS
            ],
            data=df.to_dict('records'),
        
            # Table features
            page_size=12,
            page_action="native",
            sort_action="native",
            filter_action="native",
            row_selectable="single",
            selected_rows=[],
        
            # Styling
            style_table={"overflowX": "auto", "overflowY": "auto", "height": "420px"},
            fixed_rows={"headers": True},
            style_cell={
                "textAlign": "left",
                "padding": "6px",
                "minWidth": "110px",
                "whiteSpace": "normal",
            },
            style_header={"fontWeight": "bold"},
        ),

        html.Br(),
        html.Hr(),
    
        # Side by side Chart + Map
        html.Div(
            style={"display": "flex", "gap": "16px"},
            children=[
                html.Div(
                    style={"flex": "1"},
                    children=[
                        html.H4("Second Chart"),
                        dcc.Graph(id="pie-chart", figure=empty_figure("Preferred Animals")),
                    ],
                ),
                html.Div(
                    id="map-id",
                    style={"flex": "1"},
                    children=[
                        html.H4("Geolocation Chart"),
                        dl.Map(
                            id="map",
                            center=(30.2672, -97.7431),  # Austin Default
                            zoom=10,
                            style={"width": "100%", "height": "520px"},
                            children=[dl.TileLayer(), dl.LayerGroup(id="map-layer"),
                            ],
                        ),
                    ],
                ),
             ],
        ),
    ]
)

# ---------------------------------
# Rescue constants + query helpers
# ---------------------------------

WATER_BREEDS = ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]
WATER_SEX = "Intact Female"
WATER_AGE_MIN = 26
WATER_AGE_MAX = 156

MOUNTAIN_BREEDS = [
    "German Shepherd",
    "Alaskan Malamute",
    "Old English Sheepdog",
    "Siberian Husky",
    "Rottweiler",
]
MOUNTAIN_SEX = "Intact Male"
MOUNTAIN_AGE_MIN = 26
MOUNTAIN_AGE_MAX = 156

DISASTER_BREEDS = [
    "Doberman Pinscher",
    "German Shepherd",
    "Golden Retriever",
    "Bloodhound",
    "Rottweiler",
]
DISASTER_SEX = "Intact Male"
DISASTER_AGE_MIN = 20
DISASTER_AGE_MAX = 300


def breed_or_regex(breeds):
    """
    Use case insensitive regex so mix names still match.
    For example 'Doberman Pinsch' will still match the Doberman filter.
    """
    ors = []
    for b in breeds:
        token = "Doberman" if b.lower().startswith("doberman") else b
        ors.append({"breed": {"$regex": token, "$options": "i"}})
    return {"$or": ors}


def build_query(filter_value):
    """Build the MongoDB query for wach rescue option."""
    if filter_value == "WATER":
        return {
            "animal_type": "Dog",
            "sex_upon_outcome": WATER_SEX,
            "age_upon_outcome_in_weeks": {"$gte": WATER_AGE_MIN, "$lte": WATER_AGE_MAX},
            **breed_or_regex(WATER_BREEDS),
        }

    if filter_value == "MOUNTAIN":
        return {
            "animal_type": "Dog",
            "sex_upon_outcome": MOUNTAIN_SEX,
            "age_upon_outcome_in_weeks": {"$gte": MOUNTAIN_AGE_MIN, "$lte": MOUNTAIN_AGE_MAX},
            **breed_or_regex(MOUNTAIN_BREEDS),
        }

    if filter_value == "DISASTER":
        return {
            "animal_type": "Dog",
            "sex_upon_outcome": DISASTER_SEX,
            "age_upon_outcome_in_weeks": {"$gte": DISASTER_AGE_MIN, "$lte": DISASTER_AGE_MAX},
            **breed_or_regex(DISASTER_BREEDS),
        }

    return {}  # RESET / unfiltered


def docs_to_df(docs):
    """Convert Mongo docs to a clean DataFrame that works well with Dash."""
    df_local = pd.DataFrame.from_records(docs)
    if df_local.empty:
        # Keep the column list consistent for Dash even when there are zero results
        return pd.DataFrame(columns=TABLE_COLUMNS)

    if "_id" in df_local.columns:
        df_local = df_local.drop(columns=["_id"])

    # Helpful for the map
    for col in ["location_lat", "location_long", "age_upon_outcome_in_weeks"]:
        if col in df_local.columns:
            df_local[col] = pd.to_numeric(df_local[col], errors="coerce")

    # Reindex to the stable table columns to avoid any issues in the DataTable
    return df_local.reindex(columns=TABLE_COLUMNS)
    
#############################################
# Interaction Between Components / Controller
#############################################

@app.callback(
    Output("datatable-id", "data"),
    Output("filter-status", "children"),
    Output("datatable-id", "page_current"),
    Output("datatable-id", "selected_rows"),
    Input("rescue-filter", "value"),
)
def update_dashboard(filter_value):
    """Update the table when the user changes the rescue filter."""
    query = build_query(filter_value) if filter_value else {}
    docs = db.read(query)
    df_filtered = docs_to_df(docs)
        
    label = {
        "WATER": "Water Rescue",
        "MOUNTAIN": "Mountain / Wilderness Rescue",
        "DISASTER": "Disaster Rescue / Individual Tracking",
        "RESET": "Reset (Unfiltered)",
        None: "Reset (Unfiltered)",
    }.get(filter_value, "Reset (Unfiltered)")
    
    status = f"Active Filter: {label} - Records: {len(df_filtered)}"
            
    # Reset to page 0 and clear selection when filters change
    return df_filtered.to_dict("records"), status, 0, []


@app.callback(
    Output("pie-chart", "figure"),
    Input("datatable-id", "derived_virtual_data"),
)
def update_pie(viewData):
    """
    Build a Top 10 breeds chart based on what the user is currently viewing
    (after table filtering/sorting).
    """
    if viewData is None:
        raise PreventUpdate
        
    if not viewData:
        return empty_figure("Preferred Animals (No Data)")

    dff = pd.DataFrame.from_records(viewData)
    if "breed" not in dff.columns or dff["breed"].dropna().empty:
        return empty_figure("Preferred Animals (No Breed Data)")

    counts = dff["breed"].dropna().astype(str).value_counts()
    top = counts.head(10)
    other = int(counts.iloc[10:].sum())

    pie_df = pd.DataFrame(
        {
            "breed": list(top.index) + (["Other"] if other > 0 else []),
            "count": list(top.values) + ([other] if other > 0 else []),
        }
    )

    return px.pie(pie_df, names="breed", values="count", title="Preferred Animals (Top 10 Breeds)")


@app.callback(
    Output("datatable-id", "style_data_conditional"),
    Input("datatable-id", "selected_columns"),
)
def update_styles(selected_columns):
    """Light highlight when a column header is selected."""
    if not selected_columns:
        return []
    return [{"if": {"column_id": c}, "backgroundColor": "#D2F3FF"} for c in selected_columns]


@app.callback(
    Output("map-layer", "children"),
    Input("datatable-id", "derived_virtual_data"),
    Input("datatable-id", "derived_virtual_selected_rows"),
)
def update_map(viewData, selected_rows):
    # Prevent the map from disappearing during transient updates
    if viewData is None:
        raise PreventUpdate
        
    """Update the map marker when the user selects a row in the table."""
    if not viewData:
        return []

    dff = pd.DataFrame.from_records(viewData)
    if dff.empty:
        return []
            
    row = selected_rows[0] if selected_rows else 0
    row = min(row, len(dff) - 1)

    # Default Austin center
    center = [30.2672, -97.7431]
    
    lat = dff.loc[row, "location_lat"] if "location_lat" in dff.columns else None
    lon = dff.loc[row, "location_long"] if "location_long" in dff.columns else None
    
    try:
        if pd.notna(lat) and pd.notna(lon):
            center = [float(lat), float(lon)]
    except Exception:
        pass

    breed = dff.loc[row, "breed"] if "breed" in dff.columns else "Animal"
    name = dff.loc[row, "name"] if "name" in dff.columns else "Unknown"

    return [
        dl.Marker(
            position=center,
            children=[
                dl.Tooltip(str(breed)),
                dl.Popup([html.H1("Animal Name"), html.P(str(name))]),
            ],
        )
    ]


# Run the app
app.run_server(mode="jupyterlab", debug=False)

 * Running on http://127.0.0.1:8050/ (Press CTRL+C to quit)
127.0.0.1 - - [14/Dec/2025 20:44:31] "GET /_alive_4d0df9c2-88e8-4915-99de-7294e588aabe HTTP/1.1" 200 -
127.0.0.1 - - [14/Dec/2025 20:44:32] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [14/Dec/2025 20:44:32] "GET /_dash-dependencies HTTP/1.1" 200 -
127.0.0.1 - - [14/Dec/2025 20:44:33] "GET /_dash-layout HTTP/1.1" 200 -
127.0.0.1 - - [14/Dec/2025 20:44:33] "GET /_dash-component-suites/dash/dcc/async-graph.js HTTP/1.1" 304 -
127.0.0.1 - - [14/Dec/2025 20:44:33] "GET /_dash-component-suites/dash/dash_table/async-highlight.js HTTP/1.1" 304 -
127.0.0.1 - - [14/Dec/2025 20:44:33] "POST /_dash-update-component HTTP/1.1" 204 -
127.0.0.1 - - [14/Dec/2025 20:44:33] "POST /_dash-update-component HTTP/1.1" 204 -
127.0.0.1 - - [14/Dec/2025 20:44:33] "GET /_dash-component-suites/dash/dcc/async-plotlyjs.js HTTP/1.1" 304 -
127.0.0.1 - - [14/Dec/2025 20:44:33] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [14/Dec/2025 20:44:33] "GET /_da